Ce code prend en charge un dossier de sortie de la boucle Calcul pour toutes les données sauf celle de citations de brevet, et renvoie une partition du flux de R&D dans un panel long avec 4 colonnes : "rd_same_sector_same_country", "rd_same_sector_diff_country", "rd_diff_sector_same_country", "rd_diff_sector_diff_country".

À modifier :
- chemin du dossier
- métric
- base

On mergeant tous ces blocs sur 'Region', 'Sector' & 'annee' + les données de production, export, import & demande finale, et on obtient panel_long.csv

Version 2.0 : lit les résultats du run choisi dans `project_settings.py` (RD_UNIT, DIMENSION, TECH) et écrit le panel dans `runs/<run>/panel_estimation/`.

In [48]:
import pandas as pd
from pathlib import Path
import numpy as np
import os
import importlib
import project_settings
importlib.reload(project_settings)
from project_settings import PATH_WRK, BASE, METRIC_SPE, YEARS, SRC_DIR, RESULTS_DIR, RUN_DIR, CHOC, CHOC_SCENARIO, AGG_SUFFIX, METRIC_WANTED
print(f"CHOC = {CHOC}" + (f" (scenario {CHOC_SCENARIO}): matrices *{AGG_SUFFIX}.csv" if CHOC else ": baseline matrices *_agg.csv"))

CHOC = False: baseline matrices *_agg.csv


In [49]:
def load_your_matrix(metric: str, year: int, base: str) -> pd.DataFrame:
    """
    Charge les matrices multiIndex de niveau 2 de nom : f"{metric}_{base}_{year}.csv, qui sont exportées par le code boucle.
    Supprime le niveau level.
    """

    phys = ["FIGARO", "EXIOBASE", "GTAP", "OECD_ICIO", "CAPITAL"]

    file_path = Path(SRC_DIR + f"{BASE}/output_loop_{BASE}_{year}/{metric}_{base}_{year}{AGG_SUFFIX}.csv")   # SRC_DIR: folder of the run chosen in project_settings; AGG_SUFFIX: _agg (baseline) or _agg_choc (CHOC = True)

    if not file_path.exists():
            raise FileNotFoundError(f"Fichier introuvable : {file_path}")
    if base in phys:
        matrix = pd.read_csv(
            file_path,
            sep=";",
            header=[0, 1, 2],
            index_col=[0, 1, 2],
        )

        matrix.index = matrix.index.droplevel("Level")
        matrix.columns = matrix.columns.droplevel("Level")
    else:
        matrix = pd.read_csv(
            file_path,
            sep=";",
            header=[0, 1],
            index_col=[0, 1, 2],
        )

        if "Year" in matrix.index.names or matrix.index.nlevels == 3:
            matrix.index = matrix.index.droplevel(0)

    matrix.index.names = ["Region", "Sector"]
    matrix.columns.names = ["Region", "Sector"]

    return matrix

In [50]:
def compute_4_blocks(matrix: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    matrix : DataFrame carré, index = origine (Region, Sector),
             colonnes = destination (Region, Sector)

    Retourne un DataFrame indexé par (Region, Sector) = destination,
    avec 4 colonnes qui partitionnent totalement le flux reçu.
    """
    regions_rows = matrix.index.get_level_values("Region").to_numpy()
    sectors_rows = matrix.index.get_level_values("Sector").to_numpy()
    regions_cols = matrix.columns.get_level_values("Region").to_numpy()
    sectors_cols = matrix.columns.get_level_values("Sector").to_numpy()

    same_region = regions_rows[:, None] == regions_cols[None, :]
    same_sector = sectors_rows[:, None] == sectors_cols[None, :]

    values = matrix.to_numpy()

    def block_sum(mask):
        return pd.Series(np.where(mask, values, 0.0).sum(axis=0), index=matrix.columns)

    out = pd.DataFrame({
        f"rd_same_sector_same_country_{metric}": block_sum(same_sector & same_region),
        f"rd_same_sector_diff_country_{metric}": block_sum(same_sector & ~same_region),
        f"rd_diff_sector_same_country_{metric}": block_sum(~same_sector & same_region),
        f"rd_diff_sector_diff_country_{metric}": block_sum(~same_sector & ~same_region),
    })
    out.index.names = ["Region", "Sector"]
    return out

Fonction pour ajouter les imports, exports, etc

In [51]:
def add_trade_and_production(path_wrk: str, year: list, base: str, df: pd.DataFrame) -> pd.DataFrame:
    def load_and_melt(name, value_name, id_vars):
        path = SRC_DIR + f"{base}/output_loop_{base}/{name}_{base}_agg.csv"
        wide = pd.read_csv(path, header=0, sep=";")
        year_cols = [c for c in wide.columns if c not in id_vars]
        long = wide.melt(
            id_vars=id_vars,
            value_vars=year_cols,
            var_name="year",
            value_name=value_name,
        )
        long["year"] = long["year"].astype(int)
        return long

    exports = load_and_melt("exports", "exports", ["Region", "Sector"])
    imports_final_demand = load_and_melt("imports_final_demand", "imports_final_demand", ["Region"])
    imports_intermediate = load_and_melt("imports_intermediate", "imports_intermediate", ["Region", "Sector"])
    production = load_and_melt("production", "production", ["Region", "Sector"])

    df_complete = df.reset_index()

    for y in year:
        if y not in exports["year"].values:
            raise AssertionError(f"Year {y} not in {base}_exports")
        if y not in imports_final_demand["year"].values:
            raise AssertionError(f"Year {y} not in {base}_imports_final_demand")
        if y not in imports_intermediate["year"].values:
            raise AssertionError(f"Year {y} not in {base}_imports_intermediate")
        if y not in production["year"].values:
            raise AssertionError(f"Year {y} not in {base}_production")

    df_complete = df_complete.merge(exports, on=["Region", "Sector", "year"], how="left")
    df_complete = df_complete.merge(imports_final_demand, on=["Region", "year"], how="left")
    df_complete = df_complete.merge(imports_intermediate, on=["Region", "Sector", "year"], how="left")
    df_complete = df_complete.merge(production, on=["Region", "Sector", "year"], how="left")

    df_complete = df_complete[df_complete["year"].isin(year)]
    df_complete = df_complete.set_index(["Region", "Sector", "year"])

    return df_complete

In [52]:
METRIC = METRIC_SPE[f"{BASE}_dir"]
panel_rows = []
for year in YEARS:
    matrix = load_your_matrix(METRIC, year, BASE)
    df_year = compute_4_blocks(matrix, METRIC)
    df_year["year"] = year
    panel_rows.append(df_year.reset_index())
panel = pd.concat(panel_rows, ignore_index=True).set_index(["Region", "Sector", "year"])

if BASE not in ["Patents_Citation", "FDI"]:
    METRIC = METRIC_SPE[f"{BASE}_embod"]
    panel_rows2 = []
    for year in YEARS:
        matrix2 = load_your_matrix(METRIC, year, BASE)
        df_year2 = compute_4_blocks(matrix2, METRIC)
        df_year2["year"] = year
        panel_rows2.append(df_year2.reset_index())
    panel2 = pd.concat(panel_rows2, ignore_index=True).set_index(["Region", "Sector", "year"])

    panel = pd.concat([panel, panel2], axis=1)

    if BASE == "CAPITAL" and METRIC_WANTED == "embod_investment":
        metric_investment = METRIC_SPE["CAPITAL_embod_investment"]
        panel_rows_investment = []

        for year in YEARS:
            matrix_investment = load_your_matrix(
                metric_investment, year, BASE
            )
            df_investment = compute_4_blocks(
                matrix_investment, metric_investment
            )
            df_investment["year"] = year
            panel_rows_investment.append(df_investment.reset_index())

        panel_investment = pd.concat(
            panel_rows_investment, ignore_index=True
        ).set_index(["Region", "Sector", "year"])

        panel = pd.concat([panel, panel_investment], axis=1)

    panel = add_trade_and_production(PATH_WRK, YEARS, BASE, panel)

os.makedirs(RESULTS_DIR + "panel_estimation", exist_ok=True)
panel.to_csv(RESULTS_DIR + f"panel_estimation/panel_estimation_{BASE}.csv")
print(f"run {RUN_DIR} | {BASE} | years {YEARS} -> {RESULTS_DIR}panel_estimation/panel_estimation_{BASE}.csv")

run SRDQ (c,s) | CAPITAL | years [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020] -> D:/ENTICE/rendu_fin_stage/runs/SRDQ (c,s)/panel_estimation/panel_estimation_CAPITAL.csv
